# Secondary metabolites and liquid aroma data review

This notebook visualizes the data now available for nitrogen fractions, pyruvic acid, acetaldehyde, acetic acid, and liquid aroma measurements. The purpose is model-structure selection, not parameter estimation.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / 'results').exists() and (ROOT / 'fermentation_model').exists():
    ROOT = ROOT / 'fermentation_model'
RESULTS = ROOT / 'results/secondary_metabolite_data_review'
summary = pd.read_csv(RESULTS / 'observation_summary.csv')
batch_summary = pd.read_csv(RESULTS / 'batch_variable_summary.csv')
summary

## Observation availability

Use this table to decide which variables can support dynamic calibration and which should remain diagnostic outputs or regularized states.

In [ ]:
display(summary.sort_values(['medium', 'variable']))
availability = summary.pivot(index='variable', columns='medium', values='n_obs').fillna(0).astype(int)
display(availability)
display(batch_summary.head(20))

## Immediate interpretation

- Pyruvate has enough repeated trajectories to be considered a transient state.
- Acetaldehyde is available in natural and historical batches, but not synthetic, so it is useful for validation and regularized fitting rather than unconstrained expansion.
- Acetic acid is currently natural-only, so it should enter as a low-dimensional auxiliary state.
- Liquid aromas are currently historical-only in this merged review; new natural aroma data should be appended before model-based aroma DOE.
- PAN and ammonium have distinct behavior and should be retained as separate nitrogen pools in the next model version.

## YAN component consistency

The loader keeps total `YAN`, `PAN`, and `NH4` separately. The residual `YAN - (PAN + NH4)` is useful for detecting unit, assay, or preprocessing differences.

In [ ]:
yan = pd.read_csv(RESULTS / 'yan_component_consistency.csv')
yan.dropna(subset=['YAN_component_residual_mg_l']).groupby('medium')['YAN_component_residual_mg_l'].describe()

## Per-batch plots

The following figures show sugars, biomass, nitrogen fractions, secondary metabolites, ethanol/glycerol, and liquid aromas by batch.

In [ ]:
for png in sorted((RESULTS / 'plots').glob('secondary_*.png'))[:12]:
    print(png.name)
    display(Image(filename=str(png)))

## Variable overlays

Overlay plots help identify non-Monod behavior, transient peaks, and medium-specific differences.

In [ ]:
for png in sorted((RESULTS / 'plots').glob('overlay_*.png')):
    print(png.name)
    display(Image(filename=str(png)))

## Modeling notes

See `secondary_metabolite_modeling_notes.md` for the proposed ODE treatment of pyruvate, acetaldehyde, acetate, net liquid aroma outputs, and oxygen.

In [ ]:
print((RESULTS / 'secondary_metabolite_modeling_notes.md').read_text(encoding='utf-8'))